In [ ]:
!pip install lightgbm

# Идея: Кластеризация молекул

Почему отказались: модель преувеличивала SI и не давала особого прироста к точности. Качество кластеризации было невысоким, по многим оценкам в данных было 2-4 плохо разделимых кластера. Итоговые конвейеры показывали куда более низкую RMSE на тренировочных моделях

# Описание идеи

Поскольку у нас нет исходных SMILES (чтобы построить Morgan Fingerprints), можно восстановить «химическое родство» молекул математически. Применим алгоритм K-Means (кластеризацию). Он разобьет все 1000 молекул на 10 «химических семейств» на основе сходства их 214 дескрипторов.Затем посчитаем среднюю токсичность и среднюю активность для каждого семейства молекул в train и передадим эти средние значения в test. Для CatBoost это станет супер-фичей: модель сразу поймет, к какому «классу веществ» относится молекула и какая у этого класса средняя базовая активность в реальном мире

**План**
- Объединяем признаки train и test, чтобы алгоритм K-Means одинаково выделил 10 скрытых структурных кластеров (семейств молекул).
- Считаем средние значения IC50 и CC50 для каждого кластера по обучающей выборке и добавляем их как новые признаки.
- Обучаем финальный ансамбль CatBoost на расширенном наборе данных в оригинальном масштабе с сохранением предыдущей победной пропорции блендинга SI (75% математики + 25% модели).

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from lightgbm import LGBMRegressor
from sklearn.cluster import KMeans

In [5]:
# Загружаем данные
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

In [6]:
IC50_PHYSICAL_MIN = train['IC50, mM'].min() # 0.003517

In [7]:
target_cols_all = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in test.columns if col not in target_cols_all and col != 'index']

# Заполняем пропуски в сырых данных для корректной работы K-Means
X_train_raw = train[feature_cols].fillna(train[feature_cols].median())
X_test_raw = test[feature_cols].fillna(train[feature_cols].median())


In [8]:
# === ШАГ 1: КЛАСТЕРИЗАЦИЯ СТРУКТУРЫ МОЛЕКУЛ ===
# Объединяем признаки для поиска общих химических семейств
X_all = pd.concat([X_train_raw, X_test_raw], axis=0)

# Разбиваем молекулы на 10 скрытых структурных кластеров
kmeans = KMeans(n_clusters=10, random_state=42)
all_clusters = kmeans.fit_predict(X_all)

# Возвращаем метки кластеров в датасеты
train['cluster'] = all_clusters[:len(train)]
test['cluster'] = all_clusters[len(train):]

In [9]:
# === ШАГ 2: TARGET ENCODING ПО КЛАСТЕРАМ ===
# Считаем средние биологические показатели для каждого семейства молекул в train
cluster_means = train.groupby('cluster')[['IC50, mM', 'CC50, mM']].mean().reset_index()
cluster_means.columns = ['cluster', 'cluster_mean_IC50', 'cluster_mean_CC50']

# Присоединяем эти «инсайдерские» химические признаки к train и test
X_train_raw = X_train_raw.merge(train[['cluster']], left_index=True, right_index=True)
X_train_raw = X_train_raw.merge(cluster_means, on='cluster', how='left').drop(columns=['cluster'])

X_test_raw = X_test_raw.merge(test[['cluster']], left_index=True, right_index=True)
X_test_raw = X_test_raw.merge(cluster_means, on='cluster', how='left').drop(columns=['cluster'])

# Обновляем список фичей (теперь у нас добавились кластерные средние)
updated_features = X_train_raw.columns.tolist()

# Готовим финал
submission_final = pd.DataFrame({'index': test['index']})
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
pure_test_preds = {}

In [11]:
# Обучаем модели на расширенных признаках
for target in target_cols_all:
    y_train = train[target]
    
    # Отбор топ-100 лучших признаков (увеличили емкость под новые фичи)
    selector = SelectKBest(score_func=f_regression, k=100)
    X_train_selected = pd.DataFrame(selector.fit_transform(X_train_raw, y_train))
    X_test_selected = pd.DataFrame(selector.transform(X_test_raw))
    
    test_preds = np.zeros(len(test))
    rmse_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_selected, y_train)):
        X_tr, y_tr = X_train_selected.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train_selected.iloc[val_idx], y_train.iloc[val_idx]
        
        model = CatBoostRegressor(
            iterations=1800,
            learning_rate=0.012,
            depth=6,
            loss_function='RMSE',
            l2_leaf_reg=5,
            random_seed=42,
            verbose=False
        )
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=100, verbose=False)
        val_preds = model.predict(X_va)
        rmse_scores.append(root_mean_squared_error(y_va, val_preds))
        
        test_preds += model.predict(X_test_selected) / kf.n_splits
        
    print(f"Кластерный CatBoost [{target}] -> Локальный CV RMSE: {np.mean(rmse_scores):.4f}")
    pure_test_preds[target] = test_preds


Кластерный CatBoost [IC50, mM] -> Локальный CV RMSE: 314.1822
Кластерный CatBoost [CC50, mM] -> Локальный CV RMSE: 435.5999
Кластерный CatBoost [SI] -> Локальный CV RMSE: 593.4381


In [12]:
# Шаг 3: Сборка финального ответа с физическими ограничениями
submission_final['IC50'] = np.clip(pure_test_preds['IC50, mM'], 0.0, None)
submission_final['CC50'] = np.clip(pure_test_preds['CC50, mM'], 0.0, None)

# взвешенный блендинг стратегий для SI
protected_ic50 = np.clip(submission_final['IC50'], IC50_PHYSICAL_MIN, None)
si_strategy_math = submission_final['CC50'] / protected_ic50
si_strategy_model = np.clip(pure_test_preds['SI'], 0.0, None)

submission_final['SI'] = 0.75 * si_strategy_math + 0.25 * si_strategy_model
submission_final['SI'] = np.clip(submission_final['SI'], 0.0, train['SI'].max())

print("\nПроверка средних значений кластерного ансамбля:")
print(submission_final[['IC50', 'CC50', 'SI']].mean())

# Запись финального файла
submission_final.to_csv('cluster_target_submission.csv', index=False)
print("\n[ГОТОВО] Файл 'cluster_target_submission.csv' успешно создан.")


Проверка средних значений кластерного ансамбля:
IC50    244.818821
CC50    642.073289
SI       24.658313
dtype: float64

[ГОТОВО] Файл 'cluster_target_submission.csv' успешно создан.
